<a href="https://colab.research.google.com/github/Yashcode007/masters_in_ai/blob/deep_learning_wb_84520/residual_connections_4_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

torch.nn.Sequential(*layers) creates a neural network where the output of one layer is automatically passed as the input to the next layer.

Let's break it down.

**What is layers?**

Usually, layers is a Python list containing PyTorch layer objects.

layers = [
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
]

**What does nn.Sequential do?**

It creates one module that chains together all the layers.

Internally, it behaves roughly like this:

class Sequential(nn.Module):
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

So during the forward pass,

x = layer1(x)
x = layer2(x)
x = layer3(x)
...

Each layer receives the output of the previous layer.

In [2]:
import torch
from typing import Any

In [8]:
class MyModelLN(torch.nn.Module):
  def __init__(self,layer_size = [512,512,512]) -> None:
    super().__init__()
    layers = []
    layers.append(torch.nn.Flatten())
    c = 128*128*3
    for s in layer_size():
      layers.append(torch.nn.Linear(c,s))
      layers.append(torch.nn.LayerNorm(s))
      layers.append(torch.nn.ReLu())
      c=s
    layers.append(torch.nn.Linear(c,102,bias=False))
    self.model = torch.nn.Sequential(*layers)

  def forward(self,x)-> Any:
    return self.model(x)

Residual Networks are not sequential anymore . They have skip connections in them that sort of skip a bunch of sequential layers

In [25]:
class MyModelLN(torch.nn.Module):
  class Block(torch.nn.Module):
    def __init__(self,in_channels, out_channels)-> None:
      super().__init__()
      self.linear = torch.nn.Linear(in_channels, out_channels)
      self.norm = torch.nn.LayerNorm(out_channels)
      self.relu = torch.nn.ReLU()
      #This is done because the input channel and output channel sizes are not equal .
      #Hence , when they are not equal , we make the input channel pass through Linear layer which makes it equal to the output channel

      if in_channels != out_channels:
        self.skip = torch.nn.Linear(in_channels, out_channels)
      else:
        self.skip = torch.nn.Identity()

    def forward(self,x) -> Any:
      y = self.relu(self.norm(self.linear(x)))
      # + y is the residual connection main learning
      return self.skip(x) + y

  def __init__(self, layer_size=[512,512,512]) -> None:
    super().__init__()
    layers = []
    layers.append(torch.nn.Flatten())
    c = 128*128*3
    for s in layer_size:
      layers.append(self.Block(c,s))
      c = s
    layers.append(torch.nn.Linear(c,102,bias=False))
    self.model = torch.nn.Sequential(*layers)

  def forward(self,x) -> Any:
    return self.model(x)

x = torch.randn(10,3,128,128)
net = MyModelLN([512]*4)

In [26]:
print(net(x))

tensor([[-1.1692e-01,  1.8182e+00, -1.3747e-01,  ..., -5.5888e-01,
          1.1489e+00,  3.7180e-01],
        [ 9.1426e-01, -3.0718e-01, -1.4514e+00,  ...,  1.7372e-02,
          2.1592e+00,  1.5708e+00],
        [-1.2271e-01,  8.8983e-01,  3.3200e-05,  ..., -6.1858e-01,
          1.5511e+00,  1.9912e+00],
        ...,
        [-1.0287e+00,  1.7609e+00, -1.6217e+00,  ..., -1.0463e+00,
          1.3078e+00,  1.4327e+00],
        [-3.6970e-01,  2.3206e-02, -2.1511e+00,  ..., -9.4393e-01,
          2.8021e+00,  1.6774e+00],
        [ 4.9752e-01, -1.4654e-01, -1.3780e+00,  ..., -1.1167e+00,
          1.4121e+00,  2.3065e+00]], grad_fn=<MmBackward0>)
